# Tutorial 5: Watching a real tensor compiler

Follow the same computation through a C reference, a GPU kernel, and a TileLang program. Inspect a saved compiler trace to see how tile-level operations become lower-level code.

**What changes during compilation, and what must stay the same?**

This walkthrough uses a historical Metal trace and runs on a CPU runtime.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jianwen-zhu/2026-fall-ece467-jzhu/blob/gh-pages/tutorial5.ipynb)

## Setup

Run these setup cells once, then work through the examples in order. In Colab, use a CPU runtime. This walkthrough uses Python's standard library; no compiler or extra packages are needed.

In [ ]:
#@title Restore companion files
import base64, io, zipfile
from pathlib import Path

payload = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as files:
    for name in files.namelist():
        path = Path(name)
        if path.is_absolute() or ".." in path.parts:
            raise ValueError("Unsafe companion path")
        if not path.exists():
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(files.read(name))
print("Companion files ready; existing files preserved.")


## Explore the sources

The setup cell supplies the example sources and saved trace. The following cells display selected code so you can compare its representations.

In [ ]:
%%bash
set -euo pipefail
root="$PWD"
cd "$root"
for file in course/labs/harness.c course/labs/axpy/03_tilelang/axpy.py \
            course/labs/gemm/03_tilelang/gemm.py course/labs/gemm/04_watch/passes.txt \
            deps/libttl/include/libttl.h; do
    test -f "$file" || { echo "Missing $file: run the companion-file setup cell first." >&2; exit 1; }
done
printf 'Companion files ready: %s\nLocal files only; GPU execution has not been checked.\n' "$root"


## Let us follow one AXPY element

AXPY computes `output[i] = alpha*x[i] + y[i]`. The C reference defines that result. The GPU kernel chooses which thread computes each element.

**Before looking below:** with 256 threads per block, which block and local thread compute element 700?

We will look at just three pieces: the reference loop, the native index calculation, and the corresponding TileLang calculation. TileLang is a GPU kernel language; its Python source is compiled, while the application harness remains C.


In [ ]:
%%bash
set -euo pipefail
root="$PWD"
cd "$root"
printf 'C reference:\n'
sed -n '/^void axpy_reference(/,$p' course/labs/axpy/01_cpu/axpy_reference.c
printf '\nHandwritten CUDA:\n'
cat course/labs/axpy/02_device/axpy.cu
printf '\nTileLang:\n'
cat course/labs/axpy/03_tilelang/axpy.py


Element 700 is computed by **block 2, thread 188**. Notice that the formula survives the change of language.

Now change the block size from 256 to 128 threads for 1024 elements. We need eight blocks, and the grid, thread count, `T.Parallel`, index stride, and `TTL_LAUNCH` must agree. Changing one number is not enough.

What has not changed is the check. **libttl**, the Tiny Tensor Library, lets the C runner load a module, launch it, synchronize, and compare its output with the reference using `ttl_close`. The module's JSON manifest describes its tensor/scalar interface. Physical argument packaging may differ between producers without changing AXPY's meaning.

**Takeaway:** we can replace the kernel producer without replacing the application's reference or numerical check. The complete sequence is in `course/labs/runner_ttl.c`.


## GEMM makes the schedule visible

Our next example multiplies two 128×128 FP16 matrices, accumulating into FP32. For one output element, the reference computes:

```c
for (size_t inner = 0; inner < k; ++inner)
    accumulator += A[row*k + inner] * B[inner*n + column];
```

This excerpt shows the addressing; the actual reference converts FP16 input bits to FP32 before multiplying.

Let us look for **shared tiles, two barriers, and the inner product** in the handwritten kernel. Then find the corresponding tile operations in TileLang. A barrier coordinates threads before data is consumed or overwritten.


In [ ]:
%%bash
set -euo pipefail
root="$PWD"
cd "$root"
printf 'Handwritten tiled CUDA:\n'
sed -n '/    __shared__/,$p' course/labs/gemm/02_device/gemm.cu
printf '\nTileLang tile sizes and operations:\n'
sed -n '/TTL_SPECIALIZATIONS/,$p' course/labs/gemm/03_tilelang/gemm.py


For local thread `(x,y)=(3,5)`, in block (0,0), the second k tile loads **A[5,19] and B[21,3]**. The first barrier makes the loaded tile available; the second prevents premature reuse of its storage. Do not remove a barrier experimentally: a race that happens to pass is still incorrect.

In TileLang, `a_shared` is 64×32 FP16, `b_shared` is 32×64 FP16, and the accumulator is 64×64 FP32 distributed across threads. Each shared tile is 4096 bytes; three pipeline stages for both account for 24576 shared bytes.

**What should we notice?** The source has already chosen tile sizes, memory scopes, and a pipeline depth. The compiler realizes that schedule; this example does not show it discovering every scheduling decision automatically. The handwritten 16×16 schedule and the TileLang schedule need not be identical to compute the same answer.


## Where did the tile operation go?

Stage `04_watch` uses the **same GEMM source and manifest as stage 03**. It runs the complete compiler, builds the generated kernel, and checks it with the same harness. It saves a few intermediate representations (**IRs**) so we can observe the lowering.

We will concentrate on **one `LowerTileOp` diff**, not tour every compiler pass. A diff compares the representation immediately before and after a pass: removed lines begin with `-`, added lines with `+`.

**Predict:** what might replace a tile-level `T.gemm`? Look for a lower-level matrix operation or backend helper, not necessarily a single scalar multiplication.

The next cell displays the bundled historical Metal diff (2026-08-28). It does not run compilation or certify the current build. CUDA uses different backend operations. If there is more than one occurrence, the filename identifies which one we are viewing; use the adjacent before/after files for context.


In [ ]:
%%bash
set -euo pipefail
root="$PWD"
cd "$root"
selected=trace/selected
if [ ! -f "$selected/00_tile_program.tir" ]; then
    echo 'Run the companion-file setup cell to restore the trace.' >&2
    exit 1
fi
echo 'Historical Metal trace (2026-08-28), not a fresh CUDA result.'
found=0
for file in "$selected"/*LowerTileOp*.diff; do
    [ -f "$file" ] || continue
    printf '%s (first 100 lines)\n' "$file"
    sed -n '1,100p' "$file"
    found=1
    break
done
if [ "$found" -eq 0 ]; then
    echo 'No LowerTileOp diff matched; inspect selected filenames before choosing a trace.'
    find "$selected" -maxdepth 1 -type f -print | sort
fi


Let us follow **one operation**, ignoring unrelated generated names:

1. Find its abstract form in the before file.
2. Find the replacement in the after file. What execution detail is now explicit?
3. Open the bundled `trace/selected/kernel.metal` and find the corresponding native operation or helper.

Use `trace/selected/00_tile_program.tir` as the starting point, and the selected before/after files when the displayed diff needs more context. If this occurrence has no relevant change, say so and inspect the next occurrence rather than inventing a transformation.

Other saved checkpoints expose these execution details:

| Pass | Detail to follow when needed |
| --- | --- |
| `MaterializeKernelLaunch` | Grid and threads |
| `InjectSoftwarePipeline` | Staged reduction loop |
| `LayoutInference` | Thread-distributed layouts |
| `StorageRewrite` | Physical storage reuse |

**Takeaway:** lowering changes representation while preserving computation. Correctness is checked by executing the result against the reference, not by recognizing a familiar pass name.


## Diagnose a broken launch

Consider this change: reduce AXPY's native grid from four blocks to three, leaving the problem size at 1024.

**Predict first:** which outputs are never written? They are indices **768–1023**. Those values are not necessarily zero: allocation does not initialize the output.

In a live experiment, compare the prediction with the actual failure report. Offline, explain which check should detect the missing outputs; no observed failure report is bundled. Would this be a compilation, module-validation, launch, or numerical-comparison failure? These are different observations. A valid manifest does not prove correct arithmetic, and an old output file does not prove that the latest command succeeded.

Explain these three points in your own words:

- What stayed fixed when we replaced handwritten code with TileLang?
- Which scheduling decisions were already in the source?
- What did our selected compiler pass make explicit?

If you can connect the source, the trace, and the unchanged correctness check, you have the central lesson.


## Further exploration

Choose another saved pass and identify one change in representation. Explain which computation stays the same. The GEMM source retains its TileLang attribution and MIT notice.